# Análise exploratória: LUNA16, desafio completo

Cinco figuras. A regra que decide o que entra é uma só: **cada figura responde a uma
pergunta que muda uma decisão de pré-processamento.** Figura que não muda decisão sai, por
mais bonita que seja.

Trabalhamos com os 888 exames do desafio, nos dez subsets. A primeira figura sai do
inventário em `dados/intermediario/`, e as outras quatro leem o disco direto.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import Image

sys.path.insert(0, str(Path.cwd().parent / "src"))
import config
from dataset import volumes
from visualization import eda

cfg = config.carregar()
config.fixar_semente()

figuras = cfg["caminhos"]["figuras"]
hu = (cfg["pre_processamento"]["hu_min"], cfg["pre_processamento"]["hu_max"])
alvo_mm = cfg["pre_processamento"]["espacamento_alvo"][0]
patch = cfg["candidatos"]["patch"][0]

inventario = pd.read_csv(cfg["caminhos"]["intermediario"] / "inventario_volumes.csv")
anotacoes = pd.read_csv(cfg["caminhos"]["luna16_anotacoes"])
nodulos = anotacoes[anotacoes.seriesuid.isin(set(inventario.seriesuid))]
len(inventario), len(nodulos)

## 1. Espaçamento entre fatias

Os exames foram reconstruídos com espessuras diferentes. A pergunta é se dá para tratar todos como se fossem do mesmo tamanho físico.

In [ ]:
Image(eda.espacamento_em_z(inventario, figuras / "eda_espacamento_z.png", alvo_mm))

**A decisão que isto sustenta.** Como o espaçamento em z assume dez valores distintos e vai
de 0,45 a 2,5 mm, comparar volumes sem reamostrar seria comparar coisas de tamanho físico
diferente, então reamostramos tudo para 1 mm isotrópico antes de recortar qualquer patch.

## 2. Diâmetro dos nódulos

O enunciado sugere cubos de 32 por 32 por 32 voxels. A 1 mm isotrópico isso cobre 32 mm, e o maior nódulo do desafio tem 32,27 mm. A pergunta é se isso corta a cauda.

In [ ]:
Image(eda.diametro_dos_nodulos(nodulos, figuras / "eda_diametro.png", patch, alvo_mm))

**A decisão que isto sustenta.** Um único nódulo dos 1.186 passa dos 32 mm que o patch
cobre, e a mediana é 6,4 mm, então mantemos o cubo de 32 voxels do enunciado em vez de
aumentá-lo, e registramos esse caso como perda conhecida em vez de descobri-la depois.

## 3. Nódulos por exame

Nem todo exame tem nódulo. A pergunta é o que fazer com os que não têm na hora de contar falso positivo por exame.

In [ ]:
Image(eda.nodulos_por_scan(nodulos, inventario, figuras / "eda_nodulos_por_scan.png"))

**A decisão que isto sustenta.** 287 dos 888 exames não têm nódulo elegível e entram
na avaliação do desafio mesmo assim, então a média de falso positivo por exame usa os 888
no denominador e não os 601 anotados. Usar 601 inflaria o número em quase metade.

## 4. Distribuição de HU dentro do pulmão

Calculada dentro da máscara que o desafio entrega, não no volume inteiro: o pulmão é uma fração pequena da imagem e o resto é ar externo, mesa e parede torácica. A pergunta é onde cortar a janela de intensidade.

In [ ]:
import SimpleITK as sitk

amostra = inventario.sample(30, random_state=cfg["seed"])
valores = []
for linha in amostra.itertuples():
    caminho = volumes.caminho_do_volume(cfg["caminhos"]["luna16"], linha.seriesuid, cfg["selecao"]["subsets"])
    mascara = cfg["caminhos"]["luna16_mascaras"] / f"{linha.seriesuid}.mhd"
    volume = sitk.GetArrayFromImage(sitk.ReadImage(str(caminho)))
    pulmao = sitk.GetArrayFromImage(sitk.ReadImage(str(mascara))) > 0
    valores.append(volume[pulmao])

Image(eda.histograma_de_hu(np.concatenate(valores), figuras / "eda_hu.png", hu, len(valores)))

**A decisão que isto sustenta.** A janela de -1000 a 400 HU guarda 98,3% dos voxels
dentro da máscara de pulmão, medido em 30 volumes sorteados, então cortamos nela. O que fica
de fora é osso, calcificação e metal, e a cauda passa de 3.000 HU: sem o corte, um voxel
desses domina a escala do patch inteiro na hora de normalizar.

## 5. Desbalanceamento dos candidatos

O `candidates_V2.csv` é a entrada do baseline obrigatório. A pergunta é o que a proporção de positivos faz com a escolha de métrica.

In [ ]:
candidatos = pd.read_csv(cfg["caminhos"]["luna16_candidatos"])
nossos = candidatos[candidatos.seriesuid.isin(set(inventario.seriesuid))]
Image(eda.desbalanceamento_dos_candidatos(nossos, figuras / "eda_desbalanceamento.png"))

**A decisão que isto sustenta.** Nos 754.975 candidatos do desafio, 1.557 são nódulo,
ou 0,2062%, e responder sempre negativo acerta 99,79%. É a razão de a métrica ser a curva
FROC e não acurácia, e é o que obriga o treino a reamostrar as classes em vez de usar a
proporção natural do arquivo.